### Análise Exploratória de Dados (EDA)

O objetivo desta etapa é explorar os dados, identificar padrões relevantes e responder às seguintes perguntas de negócio e analíticas:

- Qual é a distribuição da variável-alvo?
- Quais são as principais correlações e relações entre as variáveis ​​(features)?
- Quais testes estatísticos podem fornecer insights relevantes para a seleção de variáveis ​​e a modelagem?
- Qual é o impacto financeiro das fraudes nos diferentes segmentos de transação?
- Quais combinações de dispositivo e identidade são os indicadores mais fortes de fraude?
- Quais padrões de valor da transação diferenciam transações fraudulentas de legítimas?
- Quais variáveis ​​anônimas (V1–V339) possuem maior poder preditivo para a detecção de fraudes?
- Qual é o limiar de decisão ideal para a implementação em produção?

In [0]:
dbutils.library.restartPython()

In [0]:
dados = spark.sql("select * from fraud_detection_dev.silver.fraud_data_clean")

In [0]:
display(dados)

### Qual é a distribuição da variável-alvo?

In [0]:
from pyspark.sql import functions as F
import matplotlib.pyplot as plt
import seaborn as sns

# ============================================================
# 1. Distribuição da variável-alvo - PySpark
# ============================================================

target_counts = (
    dados
    .groupBy("target")
    .agg(
        F.count("*").alias("count"),
        F.sum("TransactionAmt").alias("total_amount")
    )
)

total = dados.count()

target_dist = (
    target_counts
    .withColumn(
        "percentage",
        F.round((F.col("count") / F.lit(total)) * 100, 2)
    )
    .orderBy("target")
)

display(target_dist)

In [0]:
import matplotlib.pyplot as plt
import seaborn as sns

# Apenas o resultado agregado
target_pd = target_dist.toPandas()

legitimate = target_pd.loc[target_pd["target"] == 0].iloc[0]
fraud = target_pd.loc[target_pd["target"] == 1].iloc[0]

sns.set_theme(style="white")

fig, ax = plt.subplots(figsize=(10, 3.2))

# ------------------------------------------------------------
# Barra de composição
# ------------------------------------------------------------

ax.barh(
    y=0,
    width=legitimate["percentage"],
    left=0,
    height=0.32,
    color="#334155"
)

ax.barh(
    y=0,
    width=fraud["percentage"],
    left=legitimate["percentage"],
    height=0.32,
    color="#E5484D"
)

# ------------------------------------------------------------
# Rótulos
# ------------------------------------------------------------

# Legítimas
ax.text(
    legitimate["percentage"] / 2,
    0,
    f'{legitimate["percentage"]:.2f}%',
    ha="center",
    va="center",
    fontsize=13,
    color="white"
)

# Fraude
ax.text(
    legitimate["percentage"] - 1.5,
    0.32,
    f'{fraud["percentage"]:.2f}% Fraude',
    ha="center",
    va="bottom",
    fontsize=11,
    color="#E5484D"
)

# ------------------------------------------------------------
# Contagem de transações
# ------------------------------------------------------------

ax.text(
    0,
    -0.35,
    f'{legitimate["count"]:,.0f} transações legítimas',
    ha="left",
    va="center",
    fontsize=10,
    color="#64748B"
)

ax.text(
    100,
    -0.35,
    f'{fraud["count"]:,.0f} transações fraudulentas',
    ha="right",
    va="center",
    fontsize=10,
    color="#64748B"
)

# ------------------------------------------------------------
# Título
# ------------------------------------------------------------

ax.set_title(
    "Distribuição de Transações",
    loc="left",
    fontsize=16,
    fontweight="normal",
    color="#0F172A",
    pad=28
)

ax.text(
    0,
    1.10,
    "Distribuição das classes no conjunto de dados de detecção de fraude",
    transform=ax.transAxes,
    fontsize=10,
    color="#64748B"
)

# ------------------------------------------------------------
# Formatação minimalista
# ------------------------------------------------------------

ax.set_xlim(0, 100)
ax.set_ylim(-0.65, 0.65)

ax.set_xticks([])
ax.set_yticks([])

for spine in ax.spines.values():
    spine.set_visible(False)

plt.tight_layout()
plt.show()

### Qualidade dos Dados

In [0]:
from pyspark.sql import functions as F

# Contagem total de registros
total_count = dados.count()

# Calcula métricas de qualidade para todas as colunas
quality_metrics = []

for col_name in dados.columns:
    # Métricas base para todas as colunas
    col_stats = dados.agg(
        F.count(F.when(F.col(col_name).isNull(), 1)).alias("null_count"),
        F.countDistinct(col_name).alias("unique_count")
    ).collect()[0]
    
    null_count = col_stats["null_count"]
    unique_count = col_stats["unique_count"]
    
    metric = {
        "column": col_name,
        "null_count": null_count,
        "null_rate": round((null_count / total_count) * 100, 2),
        "unique_count": unique_count,
        "unique_rate": round((unique_count / total_count) * 100, 2)
    }
    
    quality_metrics.append(metric)

# Converte para DataFrame para visualização
quality_df = spark.createDataFrame(quality_metrics)

# Ordena por null_rate decrescente para ver as colunas mais problemáticas primeiro
quality_df = quality_df.orderBy(F.col("null_rate").desc())

display(quality_df)

### Detecção de Outliers

**Isolation Forest**

Para a detecção de outliers, será utilizado o algoritmo não supervisionado **Isolation Forest**.

Esse método é adequado para o conjunto de dados devido ao grande volume de registros e à quantidade de variáveis disponíveis. Diferente de métodos estatísticos univariados como IQR ou Z-Score, o Isolation Forest consegue identificar observações anômalas considerando o comportamento conjunto de múltiplas variáveis.

O algoritmo se baseia no princípio de que observações anômalas são mais raras e distintas em relação à maioria dos dados e, por isso, tendem a ser **isoladas com menos divisões** em árvores geradas aleatoriamente.

Neste projeto, o objetivo não é assumir que todo outlier representa fraude. A detecção de outliers será utilizada como uma técnica exploratória para identificar transações com comportamento incomum e avaliar sua relação com a variável `target`.

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.types import StructType, StructField, DoubleType, IntegerType
import pandas as pd
import numpy as np
from sklearn.ensemble import IsolationForest

# ============================================================
# 1. Seleção de variáveis numéricas
# ============================================================

exclude_cols = [
    "TransactionID",
    "target",
    "TransactionDT",
    "ingestion_timestamp",
    "dataset_type"
]

numeric_cols = [
    field.name
    for field in dados.schema.fields
    if field.dataType.typeName()
    in ["double", "float", "integer", "long", "decimal"]
    and field.name not in exclude_cols
]

# ============================================================
# 2. Preenchimento de valores ausentes com a mediana
# ============================================================

median_values = {}
for col in numeric_cols:
    median_val = dados.approxQuantile(col, [0.5], 0.01)[0]
    median_values[col] = median_val if median_val is not None else 0.0

dados_filled = dados
for col, median_val in median_values.items():
    dados_filled = dados_filled.fillna({col: median_val})

In [0]:
# ============================================================
# 3. Treinamento do Isolation Forest
# ============================================================

training_data = dados_filled.select(numeric_cols).toPandas()

if_model = IsolationForest(
    contamination=0.05,
    n_estimators=300,
    max_samples=256,
    random_state=42,
    n_jobs=-1
)

if_model.fit(training_data)

In [0]:
# ============================================================
# 4. Aplicação do modelo usando pandas UDF
# ============================================================

from pyspark.sql.functions import pandas_udf, PandasUDFType

schema = StructType([
    StructField("anomaly_score", DoubleType(), True),
    StructField("is_outlier", IntegerType(), True)
])

@pandas_udf(schema, PandasUDFType.SCALAR)
def predict_outliers(*cols):
    df = pd.DataFrame({col_name: col for col_name, col in zip(numeric_cols, cols)})
    scores = if_model.decision_function(df)
    predictions = if_model.predict(df)
    predictions = np.where(predictions == -1, 1, 0)
    return pd.DataFrame({"anomaly_score": -scores, "is_outlier": predictions})

dados_with_outliers = dados_filled.withColumn("outlier_results",predict_outliers(*numeric_cols))

dados_with_outliers_final = (
    dados_with_outliers
    .withColumn("anomaly_score", F.col("outlier_results.anomaly_score"))
    .withColumn("is_outlier", F.col("outlier_results.is_outlier"))
    .drop("outlier_results")
)

In [0]:
# ============================================================
# 5. Resumo dos outliers
# ============================================================

total = dados_with_outliers_final.count()

outlier_summary = (
    dados_with_outliers_final
    .groupBy("is_outlier")
    .agg(F.count("*").alias("count"))
    .withColumn("percentage", F.round(F.col("count") / F.lit(total) * 100, 2))
    .orderBy("is_outlier")
)

display(outlier_summary)


### As observações anômalas identificadas pelo Isolation Forest apresentam uma concentração de fraude maior que a população geral?

In [0]:
# ============================================================
# 6. Relação entre anomalias e fraude
# ============================================================

outlier_fraud_analysis = (
    dados_with_outliers_final
    .groupBy("is_outlier")
    .agg(
        F.count("*").alias("transactions"),
        F.sum("target").alias("fraud_transactions"),
        F.round(F.avg("target") * 100, 2).alias("fraud_rate_pct"),
        F.round(F.avg("anomaly_score"), 4).alias("avg_anomaly_score")
    )
)

display(outlier_fraud_analysis)

# ============================================================
# 7. Atualização do DataFrame principal
# ============================================================

dados = dados_with_outliers_final

In [0]:
import matplotlib.pyplot as plt
import seaborn as sns

# ============================================================
# Preparação dos dados
# ============================================================

outlier_pd = outlier_fraud_analysis.toPandas()

outlier_pd["group"] = outlier_pd["is_outlier"].map({
    1: "Outliers",
    0: "Transações regulares"
})

# Ordenação
outlier_pd = outlier_pd.sort_values(
    "fraud_rate_pct",
    ascending=True
)

# ============================================================
# Visualização executiva
# ============================================================

sns.set_theme(style="white")

fig, ax = plt.subplots(figsize=(9, 4))

colors = [
    "#475569" if group == "Transações regulares" else "#E5484D"
    for group in outlier_pd["group"]
]

bars = ax.barh(
    outlier_pd["group"],
    outlier_pd["fraud_rate_pct"],
    height=0.42,
    color=colors
)

# ============================================================
# Rótulos
# ============================================================

for bar, (_, row) in zip(bars, outlier_pd.iterrows()):

    width = bar.get_width()
    y = bar.get_y() + bar.get_height() / 2

    # Métrica principal
    ax.text(
        width + 0.35,
        y + 0.06,
        f"{row['fraud_rate_pct']:.2f}%",
        va="center",
        ha="left",
        fontsize=14,
        color="#0F172A"
    )

    # Informação secundária
    ax.text(
        width + 0.35,
        y - 0.10,
        f"{row['fraud_transactions']:,.0f} fraudes  •  "
        f"{row['transactions']:,.0f} transações",
        va="center",
        ha="left",
        fontsize=9,
        color="#64748B"
    )

# ============================================================
# Título e subtítulo
# ============================================================

ax.set_title(
    "Taxa de Fraude por Classificação de Anomalia",
    loc="left",
    fontsize=16,
    fontweight="normal",
    color="#0F172A",
    pad=28
)

ax.text(
    0,
    1.05,
    "A incidência de fraude é substancialmente maior entre transações anômalas",
    transform=ax.transAxes,
    fontsize=10,
    color="#64748B"
)

# ============================================================
# Formatação
# ============================================================

ax.set_xlabel("")
ax.set_ylabel("")

ax.set_xlim(
    0,
    outlier_pd["fraud_rate_pct"].max() * 1.55
)

ax.grid(False)

ax.tick_params(
    axis="y",
    length=0,
    labelsize=10,
    colors="#334155"
)

ax.set_xticks([])

for spine in ax.spines.values():
    spine.set_visible(False)

plt.tight_layout()
plt.show()

### Análise

Entre as transações identificadas como anômalas, **17,45% são fraudulentas**. Considerando a quantidade de registros e a natureza do problema, não será realizada a remoção desses valores, pois eles podem representar comportamentos naturais do fenômeno analisado.

Este estudo possui caráter exploratório, com o objetivo de compreender a relação entre valores discrepantes ou comportamentos anômalos nos dados e a ocorrência de transações fraudulentas. A remoção dessas observações poderia eliminar padrões relevantes para a identificação de fraudes.

### Quais são as principais correlações e relações entre as variáveis (features)?

In [0]:
# Remoção de variáveis
remove = ['ingestion_timestamp', 'dataset_type']
dados = dados.drop(*remove)

In [0]:
display(dados.limit(10))

In [0]:
from pyspark.ml.feature import StringIndexer
from pyspark.sql import functions as F
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt

# ============================================================
# 1. Identificação de colunas categóricas
# ============================================================

categorical_cols = [
    field.name
    for field in dados.schema.fields
    if field.dataType.typeName() == "string"
]

# ============================================================
# 2. Aplicação de Label Encoding usando StringIndexer
# ============================================================

dados_encoded = dados

for col in categorical_cols:
    indexer = StringIndexer(
        inputCol=col,
        outputCol=f"{col}_indexed",
        handleInvalid="keep"
    )
    indexer_model = indexer.fit(dados_encoded)
    dados_encoded = indexer_model.transform(dados_encoded)
    dados_encoded = dados_encoded.drop(col).withColumnRenamed(f"{col}_indexed", col)

# ============================================================
# 3. Seleção de colunas para correlação (exclui anomaly_score)
# ============================================================

exclude_corr = ["TransactionID", "TransactionDT", "anomaly_score", "is_outlier"]

corr_cols = [
    field.name
    for field in dados_encoded.schema.fields
    if field.dataType.typeName() in ["double", "float", "integer", "long", "decimal"]
    and field.name not in exclude_corr
]

# ============================================================
# 4. Cálculo da matriz de correlação
# ============================================================

corr_data = dados_encoded.select(corr_cols).toPandas()
corr_matrix = corr_data.corr()

# ============================================================
# 5. Filtragem de correlações altas com o alvo
# ============================================================

target_corr = corr_matrix["target"].drop("target").abs().sort_values(ascending=False)
top_features = target_corr.head(40)

display(top_features.to_frame(name="correlation_with_target"))

# ============================================================
# 6. Visualização da matriz de correlação
# ============================================================

top_feature_names = top_features.index.tolist() + ["target"]
corr_subset = corr_matrix.loc[top_feature_names, top_feature_names]

sns.set_theme(style="white")

fig, ax = plt.subplots(figsize=(14, 12))

mask = np.triu(np.ones_like(corr_subset, dtype=bool))

sns.heatmap(
    corr_subset,
    mask=mask,
    annot=True,
    fmt=".2f",
    cmap="coolwarm",
    center=0,
    square=True,
    linewidths=0.5,
    cbar_kws={"shrink": 0.8},
    ax=ax,
    vmin=-1,
    vmax=1
)

ax.set_title(
    "Matriz de Correlação - Top 20 Variáveis vs Alvo",
    fontsize=16,
    fontweight="normal",
    pad=20
)

plt.tight_layout()
plt.show()

**Análise**

Muitas variáveis possuem multicolinearidade. Nessa quantidade, até mesmo modelos baseados em árvore terão retrabalho ao tentar computar todas elas.

### Quais testes estatísticos podem fornecer insights relevantes para a seleção de variáveis e a modelagem?

**Teste Chi-Quadrado de Independência**

Método estatístico utilizado para verificar se existe associação entre duas variáveis categóricas.

- Hipótese nula: a variável explicativa e o target são independentes
- Hipótese alternativa: existe associação entre a variável explicativa e o target

O teste compara as frequências observadas com as frequências esperadas caso não existisse qualquer associação entre as variáveis.

**V de Cramer**

Teste estatístico que quantifica a intensidade da associação entre duas variáveis categóricas. A diferença entre ele e o teste Qui-Quadrado é que o V de Cramer realiza uma normalização da estatística, permitindo comparar variáveis mesmo quando a base possui um grande número de observações.

A normalização torna o resultado independente da escala da amostra, fazendo com que o V de Cramer assuma sempre valores entre 0 e 1.

**KS Univariado**

Adaptação do teste de Kolmogorov-Smirnov para avaliar uma única variável por vez. Este é um teste estatístico não paramétrico, ou seja, não existe a hipótese de que uma determinada variável possui algum comportamento de uma distribuição teórica. Ele vai responder à seguinte pergunta: a distribuição dessa variável é diferente entre transações legítimas e fraudulentas?


In [0]:
from pyspark.sql import functions as F
from scipy.stats import ks_2samp
import pandas as pd

# ============================================================
# 1. Seleção de variáveis contínuas
# ============================================================

exclude_ks = ["TransactionID", "TransactionDT", "target", "anomaly_score", "is_outlier"]

continuous_cols = [
    field.name
    for field in dados.schema.fields
    if field.dataType.typeName() in ["double", "float"]
    and field.name not in exclude_ks
]

# ============================================================
# 2. Cálculo da estatística KS para cada variável
# ============================================================

ks_results = []

for col in continuous_cols:
    # Obtém os dados para cada grupo da variável-alvo
    data_0 = dados.filter(F.col("target") == 0).select(col).dropna().toPandas()[col].values
    data_1 = dados.filter(F.col("target") == 1).select(col).dropna().toPandas()[col].values
    
    if len(data_0) > 0 and len(data_1) > 0:
        ks_stat, p_value = ks_2samp(data_0, data_1)
        
        ks_results.append({
            "variable": col,
            "ks_statistic": float(round(ks_stat, 4)),
            "p_value": float(round(p_value, 6))
        })

# ============================================================
# 3. Criação do DataFrame com os resultados
# ============================================================

statistical_tests_df = spark.createDataFrame(ks_results)

# Adiciona interpretação baseada no valor KS
statistical_tests_df = statistical_tests_df.withColumn(
    "interpretation",
    F.when(F.col("ks_statistic") > 0.3, "Alto poder discriminativo")
     .when(F.col("ks_statistic") > 0.1, "Poder discriminativo moderado")
     .otherwise("Baixo poder discriminativo")
)

statistical_tests_df = statistical_tests_df.orderBy(F.col("ks_statistic").desc())

# ============================================================
# 4. Exibe as 6 variáveis com maior KS
# ============================================================

top_ks_vars = statistical_tests_df.limit(6)

display(top_ks_vars)

In [0]:
from pyspark.sql import functions as F
from scipy.stats import chi2_contingency
import pandas as pd
import numpy as np

# ============================================================
# 1. Seleção de variáveis categóricas
# ============================================================

exclude_cat = ["TransactionID", "target"]

categorical_cols = [
    field.name
    for field in dados.schema.fields
    if field.dataType.typeName() in ["string", "integer", "long"]
    and field.name not in exclude_cat
    and not field.name.startswith("V")  # Exclude anonymous V variables
    and field.name not in ["TransactionDT", "anomaly_score", "is_outlier"]
]

# ============================================================
# 2. Cálculo do Chi-Quadrado e V de Cramer
# ============================================================

chi_results = []

for col in categorical_cols:
    # Cria tabela de contingência
    contingency_spark = (
        dados
        .groupBy(col, "target")
        .count()
        .groupBy(col)
        .pivot("target")
        .sum("count")
        .fillna(0)
    )
    
    contingency_pd = contingency_spark.toPandas()
    
    if contingency_pd.shape[0] > 1 and len(contingency_pd.columns) > 1:
        # Remove a coluna de agrupamento para obter apenas as contagens
        contingency_table = contingency_pd.iloc[:, 1:].values
        
        # Cálculo do Chi-Quadrado
        chi2, p_value, dof, expected = chi2_contingency(contingency_table)
        
        # Cálculo do V de Cramer
        n = contingency_table.sum()
        min_dim = min(contingency_table.shape[0] - 1, contingency_table.shape[1] - 1)
        cramers_v = np.sqrt(chi2 / (n * min_dim)) if min_dim > 0 else 0
        
        chi_results.append({
            "variable": col,
            "chi2_statistic": float(round(chi2, 4)),
            "cramers_v": float(round(cramers_v, 4)),
            "p_value": float(round(p_value, 6))
        })

# ============================================================
# 3. Criação do DataFrame e adição de interpretação
# ============================================================

chi_tests_df = spark.createDataFrame(chi_results)

# Adiciona interpretação baseada no valor do V de Cramer
chi_tests_df = chi_tests_df.withColumn(
    "interpretation",
    F.when(F.col("cramers_v") > 0.3, "Forte associação")
     .when(F.col("cramers_v") > 0.1, "Associação moderada")
     .otherwise("Associação fraca")
)

chi_tests_df = chi_tests_df.orderBy(F.col("cramers_v").desc())

display(chi_tests_df)

# ============================================================
# 4. Adição da coluna test_type e união com os resultados KS
# ============================================================

ks_with_type = statistical_tests_df.withColumn("test_type", F.lit("KS"))

chi_with_type = (
    chi_tests_df
    .withColumn("test_type", F.lit("Chi-Square"))
    .withColumn("ks_statistic", F.lit(None).cast("double"))
    .select("variable", "test_type", "ks_statistic", "chi2_statistic", "cramers_v", "p_value", "interpretation")
)

ks_with_cols = (
    ks_with_type
    .withColumn("chi2_statistic", F.lit(None).cast("double"))
    .withColumn("cramers_v", F.lit(None).cast("double"))
    .select("variable", "test_type", "ks_statistic", "chi2_statistic", "cramers_v", "p_value", "interpretation")
)

statistical_tests_df = ks_with_cols.union(chi_with_type)

### Qual é o impacto financeiro das fraudes nos diferentes segmentos de transação?

In [0]:
display(dados)

In [0]:
import matplotlib.pyplot as plt
import seaborn as sns
from pyspark.sql import functions as F


def format_currency(value):
    if value >= 1_000_000:
        return f"${value / 1_000_000:.2f}M"
    elif value >= 1_000:
        return f"${value / 1_000:.1f}K"
    return f"${value:,.0f}"


def plot_fraud_amount_by_segment(
    df,
    segment,
    top_n=10
):

    fraud_segment = (
        df
        .filter(F.col("target") == 1)
        .groupBy(segment)
        .agg(
            F.sum("TransactionAmt").alias("fraud_amount"),
            F.count("*").alias("fraud_transactions")
        )
        .filter(F.col(segment).isNotNull())
        .orderBy(F.col("fraud_amount").desc())
        .limit(top_n)
        .toPandas()
    )

    fraud_segment = fraud_segment.sort_values(
        "fraud_amount",
        ascending=True
    )

    sns.set_theme(style="white")

    fig, ax = plt.subplots(figsize=(10, 5.5))

    bars = ax.barh(
        fraud_segment[segment].astype(str),
        fraud_segment["fraud_amount"],
        height=0.48,
        color="#D9534F"
    )

    max_value = fraud_segment["fraud_amount"].max()

    # Espaço fixo para os rótulos
    label_x = max_value * 1.04

    for bar, (_, row) in zip(
        bars,
        fraud_segment.iterrows()
    ):

        ax.text(
            label_x,
            bar.get_y() + bar.get_height() / 2,
            format_currency(row["fraud_amount"]),
            va="center",
            ha="left",
            fontsize=10,
            color="#334155"
        )

    # Título
    ax.set_title(
        f"Valor de Transações Fraudulentas por {segment}",
        loc="left",
        fontsize=15,
        fontweight="normal",
        color="#0F172A",
        pad=24
    )

    ax.text(
        0,
        1.02,
        f"Top {top_n} categorias ordenadas por valor de transação fraudulenta",
        transform=ax.transAxes,
        fontsize=9.5,
        color="#64748B"
    )

    # Formatação
    ax.set_xlabel("")
    ax.set_ylabel("")

    ax.set_xticks([])

    ax.tick_params(
        axis="y",
        length=0,
        labelsize=10,
        colors="#334155",
        pad=8
    )

    ax.set_xlim(
        0,
        max_value * 1.18
    )

    ax.grid(False)

    for spine in ax.spines.values():
        spine.set_visible(False)

    plt.tight_layout()
    plt.show()


# Define os segmentos para análise
segments = ["ProductCD", "card4", "card6", "P_emaildomain", "card1"]

for segment in segments:
    plot_fraud_amount_by_segment(
        dados,
        segment,
        top_n=10
    )

### Quais combinações de dispositivo e identidade são os indicadores mais fortes de fraude?

In [0]:
import matplotlib.pyplot as plt
import seaborn as sns
from pyspark.sql import functions as F

# ============================================================
# 1. Taxa de fraude por tipo de dispositivo
# ============================================================

device_fraud_analysis = (
    dados
    .filter(F.col("DeviceType").isNotNull())
    .groupBy("DeviceType")
    .agg(
        F.count("*").alias("total_transactions"),
        F.sum("target").alias("fraud_transactions"),
        F.round(F.avg("target") * 100, 2).alias("fraud_rate_pct"),
        F.sum("TransactionAmt").alias("total_amount"),
        F.sum(F.when(F.col("target") == 1, F.col("TransactionAmt")).otherwise(0)).alias("fraud_amount")
    )
    .orderBy(F.col("fraud_rate_pct").desc())
)

print("Análise de Fraude por Tipo de Dispositivo:")
display(device_fraud_analysis)

# ============================================================
# 2. Modelos de dispositivo com maior taxa de fraude
# ============================================================

device_info_fraud = (
    dados
    .filter(F.col("DeviceInfo").isNotNull())
    .groupBy("DeviceInfo")
    .agg(
        F.count("*").alias("total_transactions"),
        F.sum("target").alias("fraud_transactions"),
        F.round(F.avg("target") * 100, 2).alias("fraud_rate_pct")
    )
    .filter(F.col("total_transactions") >= 50)  # Tamanho mínimo de amostra
    .orderBy(F.col("fraud_rate_pct").desc())
    .limit(15)
)

print("\nTop 15 Modelos de Dispositivo com Maior Taxa de Fraude (mín. 50 transações):")
display(device_info_fraud)

# ============================================================
# 3. Visualização: Taxa de fraude por tipo de dispositivo
# ============================================================

device_pd = device_fraud_analysis.toPandas()

if len(device_pd) > 0:
    sns.set_theme(style="white")
    
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
    
    # Taxa de fraude
    colors = ["#E5484D" if x > device_pd["fraud_rate_pct"].mean() else "#475569" 
              for x in device_pd["fraud_rate_pct"]]
    
    bars1 = ax1.barh(
        device_pd["DeviceType"],
        device_pd["fraud_rate_pct"],
        height=0.5,
        color=colors
    )
    
    for bar, (_, row) in zip(bars1, device_pd.iterrows()):
        width = bar.get_width()
        ax1.text(
            width + 0.1,
            bar.get_y() + bar.get_height() / 2,
            f"{row['fraud_rate_pct']:.2f}%",
            va="center",
            ha="left",
            fontsize=11,
            color="#0F172A"
        )
    
    ax1.set_title(
        "Taxa de Fraude por Tipo de Dispositivo",
        fontsize=14,
        fontweight="normal",
        pad=15
    )
    ax1.set_xlabel("")
    ax1.set_ylabel("")
    ax1.set_xticks([])
    ax1.grid(False)
    for spine in ax1.spines.values():
        spine.set_visible(False)
    
    # Volume de transações
    ax2.barh(
        device_pd["DeviceType"],
        device_pd["total_transactions"],
        height=0.5,
        color="#3B82F6"
    )
    
    for i, (_, row) in enumerate(device_pd.iterrows()):
        ax2.text(
            row["total_transactions"] + row["total_transactions"] * 0.02,
            i,
            f"{row['total_transactions']:,.0f}",
            va="center",
            ha="left",
            fontsize=11,
            color="#0F172A"
        )
    
    ax2.set_title(
        "Volume de Transações por Tipo de Dispositivo",
        fontsize=14,
        fontweight="normal",
        pad=15
    )
    ax2.set_xlabel("")
    ax2.set_ylabel("")
    ax2.set_xticks([])
    ax2.grid(False)
    for spine in ax2.spines.values():
        spine.set_visible(False)
    
    plt.tight_layout()
    plt.show()

print("\n" + "="*60)
print("Principais Insights:")
print("="*60)

### Quais padrões de valor da transação diferenciam transações fraudulentas de legítimas?

In [0]:
import matplotlib.pyplot as plt
import seaborn as sns
from pyspark.sql import functions as F
from scipy.stats import ks_2samp
import numpy as np

# ============================================================
# 1. Cálculo de estatísticas centrais por alvo
# ============================================================

stats_by_target = (
    dados
    .groupBy("target")
    .agg(
        F.count("TransactionAmt").alias("count"),
        F.round(F.mean("TransactionAmt"), 2).alias("mean"),
        F.round(F.stddev("TransactionAmt"), 2).alias("std"),
        F.round(F.expr("percentile(TransactionAmt, 0.5)"), 2).alias("median"),
        F.round(F.min("TransactionAmt"), 2).alias("min"),
        F.round(F.max("TransactionAmt"), 2).alias("max"),
        F.round(F.expr("percentile(TransactionAmt, 0.25)"), 2).alias("q1"),
        F.round(F.expr("percentile(TransactionAmt, 0.75)"), 2).alias("q3")
    )
    .withColumn(
        "target_label",
        F.when(F.col("target") == 0, "Legítima")
         .otherwise("Fraudulenta")
    )
    .select("target_label", "count", "mean", "std", "median", "min", "max", "q1", "q3")
)

print("Estatísticas Centrais por Tipo de Transação:")
print("="*80)
display(stats_by_target)

# ============================================================
# 2. Teste de Kolmogorov-Smirnov para duas amostras
# ============================================================

# Obtém os valores de transação para cada grupo
legitimate_amounts = (
    dados
    .filter(F.col("target") == 0)
    .select("TransactionAmt")
    .toPandas()["TransactionAmt"]
    .values
)

fraudulent_amounts = (
    dados
    .filter(F.col("target") == 1)
    .select("TransactionAmt")
    .toPandas()["TransactionAmt"]
    .values
)

# Executa o teste KS
ks_statistic, p_value = ks_2samp(legitimate_amounts, fraudulent_amounts)

print("\n" + "="*80)
print("Resultados do Teste de Kolmogorov-Smirnov:")
print("="*80)
print(f"Estatística KS: {ks_statistic:.4f}")
print(f"P-valor: {p_value:.6f}")
print(f"\nInterpretação: {'As distribuições são significativamente diferentes' if p_value < 0.05 else 'Não há diferença significativa'}")
print(f"Tamanho do Efeito: {'Grande' if ks_statistic > 0.3 else 'Moderado' if ks_statistic > 0.1 else 'Pequeno'}")

# ============================================================
# 3. Visualizações
# ============================================================

sns.set_theme(style="white")

fig, axes = plt.subplots(2, 2, figsize=(16, 10))

# Preparação dos dados para visualização
transaction_data = dados.select("TransactionAmt", "target").toPandas()
transaction_data["target_label"] = transaction_data["target"].map({
    0: "Legítima",
    1: "Fraudulenta"
})

# ============================================================
# Gráfico 1: Distribuição com KDE (Intervalo completo)
# ============================================================

ax1 = axes[0, 0]

for target, label, color in [(0, "Legítima", "#3B82F6"), (1, "Fraudulenta", "#E5484D")]:
    data = transaction_data[transaction_data["target"] == target]["TransactionAmt"]
    
    ax1.hist(
        data,
        bins=50,
        alpha=0.6,
        label=label,
        color=color,
        density=True,
        edgecolor="white",
        linewidth=0.5
    )
    
    # Adiciona curva KDE
    data.plot.kde(
        ax=ax1,
        color=color,
        linewidth=2.5,
        label=f"{label} (KDE)"
    )

ax1.set_title(
    "Distribuição do Valor da Transação (Intervalo Completo)",
    fontsize=13,
    fontweight="normal",
    pad=15
)
ax1.set_xlabel("Valor da Transação ($)", fontsize=11)
ax1.set_ylabel("Densidade", fontsize=11)
ax1.legend(loc="upper right", fontsize=10)
ax1.grid(True, alpha=0.2, linestyle="--")
ax1.spines["top"].set_visible(False)
ax1.spines["right"].set_visible(False)

# ============================================================
# Gráfico 2: Distribuição com KDE (Zoom: $0-$500)
# ============================================================

ax2 = axes[0, 1]

for target, label, color in [(0, "Legítima", "#3B82F6"), (1, "Fraudulenta", "#E5484D")]:
    data = transaction_data[
        (transaction_data["target"] == target) & 
        (transaction_data["TransactionAmt"] <= 500)
    ]["TransactionAmt"]
    
    ax2.hist(
        data,
        bins=50,
        alpha=0.6,
        label=label,
        color=color,
        density=True,
        edgecolor="white",
        linewidth=0.5
    )
    
    # Adiciona curva KDE
    data.plot.kde(
        ax=ax2,
        color=color,
        linewidth=2.5,
        label=f"{label} (KDE)"
    )

ax2.set_title(
    "Distribuição do Valor da Transação (Zoom: $0-$500)",
    fontsize=13,
    fontweight="normal",
    pad=15
)
ax2.set_xlabel("Valor da Transação ($)", fontsize=11)
ax2.set_ylabel("Densidade", fontsize=11)
ax2.legend(loc="upper right", fontsize=10)
ax2.grid(True, alpha=0.2, linestyle="--")
ax2.spines["top"].set_visible(False)
ax2.spines["right"].set_visible(False)

# ============================================================
# Gráfico 3: Comparação via Box Plot
# ============================================================

ax3 = axes[1, 0]

box_data = [legitimate_amounts, fraudulent_amounts]
box_labels = ["Legítima", "Fraudulenta"]
box_colors = ["#3B82F6", "#E5484D"]

bp = ax3.boxplot(
    box_data,
    labels=box_labels,
    patch_artist=True,
    widths=0.5,
    showmeans=True,
    meanprops=dict(marker="D", markerfacecolor="white", markeredgecolor="black", markersize=6)
)

for patch, color in zip(bp["boxes"], box_colors):
    patch.set_facecolor(color)
    patch.set_alpha(0.7)

for whisker in bp["whiskers"]:
    whisker.set(linewidth=1.5)

for cap in bp["caps"]:
    cap.set(linewidth=1.5)

for median in bp["medians"]:
    median.set(color="black", linewidth=2)

ax3.set_title(
    "Comparação de Box Plot do Valor da Transação",
    fontsize=13,
    fontweight="normal",
    pad=15
)
ax3.set_ylabel("Valor da Transação ($)", fontsize=11)
ax3.grid(True, alpha=0.2, linestyle="--", axis="y")
ax3.spines["top"].set_visible(False)
ax3.spines["right"].set_visible(False)

# ============================================================
# Gráfico 4: Função de Distribuição Acumulada (CDF)
# ============================================================

ax4 = axes[1, 1]

for target, label, color in [(0, "Legítima", "#3B82F6"), (1, "Fraudulenta", "#E5484D")]:
    data = transaction_data[transaction_data["target"] == target]["TransactionAmt"]
    
    sorted_data = np.sort(data)
    y = np.arange(1, len(sorted_data) + 1) / len(sorted_data)
    
    ax4.plot(
        sorted_data,
        y,
        label=label,
        color=color,
        linewidth=2.5,
        alpha=0.8
    )

ax4.set_title(
    f"Função de Distribuição Acumulada\nEstatística KS = {ks_statistic:.4f}",
    fontsize=13,
    fontweight="normal",
    pad=15
)
ax4.set_xlabel("Valor da Transação ($)", fontsize=11)
ax4.set_ylabel("Probabilidade Acumulada", fontsize=11)
ax4.legend(loc="lower right", fontsize=10)
ax4.grid(True, alpha=0.2, linestyle="--")
ax4.spines["top"].set_visible(False)
ax4.spines["right"].set_visible(False)

plt.tight_layout()
plt.show()

print("\n" + "="*80)
print("Principais Conclusões:")
print("="*80)

### Quais variáveis anônimas (V1–V339) possuem maior poder preditivo para a detecção de fraudes?

In [0]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pyspark.sql import functions as F
from matplotlib.patches import Patch

# ============================================================
# 1. Identificação de todas as variáveis anônimas V
# ============================================================

v_cols = sorted(
    [field.name for field in dados.schema.fields if field.name.startswith("V")],
    key=lambda x: int(x[1:])
)


# ============================================================
# 2. Definição da função de cálculo do IV
# ============================================================

def calculate_iv(series, target, n_bins=10):
    """
    Calcula o Information Value para uma variável contínua
    usando binning por quantis com suavização de Laplace.
    """
    df = pd.DataFrame({"value": series, "target": target}).dropna()

    if len(df) == 0:
        return 0.0

    total_good = int((df["target"] == 0).sum())
    total_bad = int((df["target"] == 1).sum())

    if total_good == 0 or total_bad == 0:
        return 0.0

    # Binning por quantis (igual frequência)
    try:
        df["bin"] = pd.qcut(df["value"], q=n_bins, duplicates="drop")
    except ValueError:
        return 0.0

    counts = df.groupby("bin", observed=True)["target"].value_counts().unstack(fill_value=0)
    good = counts.get(0, pd.Series(0, index=counts.index))
    bad = counts.get(1, pd.Series(0, index=counts.index))

    n_actual_bins = len(good)

    # Suavização de Laplace para evitar log(0) e divisão por zero
    good_pct = (good + 0.5) / (total_good + 0.5 * n_actual_bins)
    bad_pct = (bad + 0.5) / (total_bad + 0.5 * n_actual_bins)

    iv = float(np.sum((good_pct - bad_pct) * np.log(good_pct / bad_pct)))

    return iv


# ============================================================
# 3. Cálculo do IV para todas as variáveis V (processamento em lotes)
# ============================================================

batch_size = 50
iv_results = []

for i in range(0, len(v_cols), batch_size):
    batch_cols = v_cols[i : i + batch_size]

    # Converte apenas este lote para pandas
    batch_pd = dados.select("target", *batch_cols).toPandas()

    for col_name in batch_cols:
        iv = calculate_iv(batch_pd[col_name], batch_pd["target"])
        iv_results.append({
            "variable": col_name,
            "iv": float(round(iv, 4))
        })


# ============================================================
# 4. Criação do Spark DataFrame com interpretação
# ============================================================

iv_df = spark.createDataFrame(iv_results)

iv_df = iv_df.withColumn(
    "predictive_power",
    F.when(F.col("iv") < 0.02, "Não útil")
     .when(F.col("iv") < 0.1, "Preditor fraco")
     .when(F.col("iv") < 0.3, "Preditor médio")
     .when(F.col("iv") < 0.5, "Preditor forte")
     .otherwise("Suspeito / Muito forte")
)

iv_df = iv_df.orderBy(F.col("iv").desc())

display(iv_df)

# ============================================================
# 5. Estatísticas resumidas
# ============================================================

power_summary = (
    iv_df
    .groupBy("predictive_power")
    .agg(F.count("*").alias("count"))
    .orderBy(F.col("count").desc())
)

display(power_summary)

# ============================================================
# 6. Visualização - Top 20 variáveis V por IV
# ============================================================

top_iv_pd = iv_df.orderBy(F.col("iv").desc()).limit(20).toPandas()

sns.set_theme(style="white")

fig, ax = plt.subplots(figsize=(12, 7))

color_map = {
    "Suspeito / Muito forte": "#7C3AED",
    "Preditor forte": "#E5484D",
    "Preditor médio": "#F59E0B",
    "Preditor fraco": "#3B82F6",
    "Não útil": "#94A3B8"
}

colors = [color_map.get(pp, "#94A3B8") for pp in top_iv_pd["predictive_power"]]

bars = ax.barh(
    top_iv_pd["variable"][::-1],
    top_iv_pd["iv"][::-1],
    height=0.6,
    color=colors[::-1]
)

for bar, (_, row) in zip(bars, top_iv_pd[::-1].iterrows()):
    ax.text(
        bar.get_width() + 0.005,
        bar.get_y() + bar.get_height() / 2,
        f"{row['iv']:.4f}",
        va="center",
        ha="left",
        fontsize=10,
        color="#0F172A"
    )

ax.set_title(
    "Top 20 Variáveis Anônimas (V1-V321) por Information Value",
    loc="left",
    fontsize=15,
    fontweight="normal",
    color="#0F172A",
    pad=24
)

ax.text(
    0,
    1.02,
    "IV mede o poder preditivo de cada variável para detecção de fraude",
    transform=ax.transAxes,
    fontsize=10,
    color="#64748B"
)

ax.set_xlabel("Information Value", fontsize=11, color="#64748B")
ax.set_ylabel("")
ax.tick_params(axis="y", length=0, labelsize=10, colors="#334155", pad=8)
ax.tick_params(axis="x", length=0, labelsize=9, colors="#64748B")
ax.grid(True, alpha=0.15, linestyle="--", axis="x")
for spine in ax.spines.values():
    spine.set_visible(False)

# Legend
legend_elements = [
    Patch(facecolor="#7C3AED", label="Suspeito (IV >= 0.5)"),
    Patch(facecolor="#E5484D", label="Forte (0.3 <= IV < 0.5)"),
    Patch(facecolor="#F59E0B", label="Médio (0.1 <= IV < 0.3)"),
    Patch(facecolor="#3B82F6", label="Fraco (0.02 <= IV < 0.1)"),
    Patch(facecolor="#94A3B8", label="Não útil (IV < 0.02)")
]
ax.legend(handles=legend_elements, loc="lower right", fontsize=9, frameon=False)

plt.tight_layout()
plt.show()



In [0]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pyspark.sql import functions as F

# ============================================================
# 1. Seleção de variáveis com IV > 0.5
# ============================================================

high_iv_vars = (
    iv_df
    .filter(F.col("iv") > 0.5)
    .orderBy(F.col("iv").desc())
    .toPandas()
)

display(iv_df.filter(F.col("iv") > 0.5).orderBy(F.col("iv").desc()))

high_iv_cols = high_iv_vars["variable"].tolist()

# ============================================================
# 2. Taxa de nulos para variáveis com alto IV
# ============================================================

null_rates = []
for col_name in high_iv_cols:
    nr = dados.filter(F.col(col_name).isNull()).count() / dados.count() * 100
    null_rates.append({"variable": col_name, "null_rate_pct": float(round(nr, 2))})

null_df = spark.createDataFrame(null_rates).orderBy(F.col("null_rate_pct").desc())

display(null_df)

# ============================================================
# 3. Matriz de correlação entre variáveis com alto IV
# ============================================================

corr_pd = dados.select(high_iv_cols).toPandas()
corr_matrix = corr_pd.corr()

fig, ax = plt.subplots(figsize=(14, 11))

mask = np.triu(np.ones_like(corr_matrix, dtype=bool), k=1)

sns.heatmap(
    corr_matrix,
    mask=mask,
    annot=True,
    fmt=".2f",
    cmap="RdBu_r",
    center=0,
    vmin=-1,
    vmax=1,
    square=True,
    linewidths=0.5,
    ax=ax,
    annot_kws={"size": 7}
)

ax.set_title(
    "Matriz de Correlação - Variáveis com IV > 0.5",
    loc="left",
    fontsize=15,
    color="#0F172A",
    pad=20
)

plt.tight_layout()
plt.show()

# ============================================================
# 4. Comparação de distribuição: Fraude vs Legítimo (top 6)
# ============================================================

top6 = high_iv_cols[:6]

fig, axes = plt.subplots(2, 3, figsize=(18, 10))
axes = axes.flatten()

target_pd = dados.select("target", *top6).toPandas()

for i, col_name in enumerate(top6):
    ax = axes[i]

    fraud = target_pd[target_pd["target"] == 1][col_name].dropna()
    legit = target_pd[target_pd["target"] == 0][col_name].dropna()

    ax.hist(legit, bins=50, alpha=0.5, label="Legítima", color="#3B82F6", density=True)
    ax.hist(fraud, bins=50, alpha=0.5, label="Fraude", color="#E5484D", density=True)

    ax.set_title(f"{col_name} (IV={high_iv_vars.loc[high_iv_vars['variable'] == col_name, 'iv'].iloc[0]:.4f})",
                 fontsize=12, color="#0F172A")
    ax.legend(fontsize=9)
    ax.set_xlabel(col_name, fontsize=10)
    ax.set_ylabel("Density", fontsize=10)
    ax.grid(True, alpha=0.15, linestyle="--")
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)

plt.suptitle("Distribuição: Fraude vs Legítimo para as 6 Variáveis com Maior IV",
             fontsize=16, color="#0F172A", y=1.01)
plt.tight_layout()
plt.show()

# ============================================================
# 5. Pares altamente correlacionados (|r| > 0.8)
# ============================================================

pairs = []
for i in range(len(corr_matrix.columns)):
    for j in range(i + 1, len(corr_matrix.columns)):
        r = corr_matrix.iloc[i, j]
        if abs(r) > 0.8:
            pairs.append({
                "var_1": corr_matrix.columns[i],
                "var_2": corr_matrix.columns[j],
                "correlation": float(round(r, 3))
            })

if pairs:
    corr_pairs_df = spark.createDataFrame(pairs).orderBy(F.col("correlation").desc())
    display(corr_pairs_df)
else:
    print("Nenhum par com |correlação| > 0.8")

In [0]:
from scipy.stats import levene
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pyspark.sql import functions as F

# ============================================================
# Comparação de variâncias + Teste de Levene para variáveis com IV > 0.5
# ============================================================
#
# Com ~590K linhas, o p-valor de Levene rejeita H0 mesmo para
# diferenças negligenciáveis. Reportamos três sinais lado a lado:
#   - Razão de variâncias (tamanho do efeito / significância prática)
#   - Teste de Levene nos dados completos (quase sempre significativo)
#   - Teste de Levene em subamostra de 5000 por grupo (checagem de sensibilidade)
#   - Brown-Forsythe (centrado na mediana, robusto a não-normalidade)
#
# Critério de decisão: razão de variâncias, NÃO o p-valor.
#   razão < 1.5  -> variâncias praticamente iguais
#   razão 1.5-3 -> diferença moderada
#   razão > 3   -> diferença substancial
# ============================================================

high_iv_cols = high_iv_vars["variable"].tolist()

analysis_pd = dados.select("target", *high_iv_cols).toPandas()

results = []

for col_name in high_iv_cols:
    fraud = analysis_pd.loc[analysis_pd["target"] == 1, col_name].dropna()
    legit = analysis_pd.loc[analysis_pd["target"] == 0, col_name].dropna()

    var_fraud = float(fraud.var())
    var_legit = float(legit.var())
    var_ratio = max(var_fraud, var_legit) / min(var_fraud, var_legit) if min(var_fraud, var_legit) > 0 else float("inf")

    # Teste de Levene (centrado na media) nos dados completos
    stat_full, p_full = levene(legit, fraud, center="mean")

    # Brown-Forsythe (centrado na mediana, mais robusto a nao-normalidade)
    stat_bf, p_bf = levene(legit, fraud, center="median")

    # Teste de Levene em subamostra (5000 por grupo) para checar sensibilidade ao tamanho
    n_sub = min(5000, len(fraud), len(legit))
    fraud_sub = fraud.sample(n=n_sub, random_state=42)
    legit_sub = legit.sample(n=n_sub, random_state=42)
    stat_sub, p_sub = levene(legit_sub, fraud_sub, center="mean")

    if var_ratio < 1.5:
        practical = "Iguais (razão < 1.5)"
    elif var_ratio < 3:
        practical = "Dif. moderada (1.5-3)"
    else:
        practical = "Dif. substancial (> 3)"

    results.append({
        "variable": col_name,
        "var_legit": float(round(var_legit, 4)),
        "var_fraud": float(round(var_fraud, 4)),
        "var_ratio": float(round(var_ratio, 2)),
        "levene_p_full": float(round(p_full, 6)),
        "brown_forsythe_p": float(round(p_bf, 6)),
        "levene_p_subsample": float(round(p_sub, 6)),
        "practical_assessment": practical
    })

variance_df = spark.createDataFrame(results)
variance_df = variance_df.orderBy(F.col("var_ratio").desc())

display(variance_df)

# ============================================================
# Visualização: Razão de variâncias por variável
# ============================================================

var_pd = variance_df.toPandas()

sns.set_theme(style="white")

fig, ax = plt.subplots(figsize=(12, 7))

colors = ["#E5484D" if r > 3 else "#F59E0B" if r > 1.5 else "#3B82F6"
          for r in var_pd["var_ratio"]]

bars = ax.barh(
    var_pd["variable"][::-1],
    var_pd["var_ratio"][::-1],
    height=0.6,
    color=colors[::-1]
)

for bar, (_, row) in zip(bars, var_pd[::-1].iterrows()):
    ax.text(
        bar.get_width() + 0.05,
        bar.get_y() + bar.get_height() / 2,
        f"{row['var_ratio']:.2f}",
        va="center",
        ha="left",
        fontsize=10,
        color="#0F172A"
    )

ax.axvline(x=1.5, color="#F59E0B", linestyle="--", linewidth=1, alpha=0.7)
ax.axvline(x=3, color="#E5484D", linestyle="--", linewidth=1, alpha=0.7)

ax.set_title(
    "Razão de Variâncias (Fraude vs Legítimo) para Variáveis com IV > 0.5",
    loc="left",
    fontsize=15,
    color="#0F172A",
    pad=24
)
ax.text(
    0,
    1.02,
    "Razão > 1.5 indica variâncias praticamente diferentes (independente do tamanho da amostra)",
    transform=ax.transAxes,
    fontsize=10,
    color="#64748B"
)
ax.set_xlabel("Razão de Variâncias (max / min)", fontsize=11, color="#64748B")
ax.set_ylabel("")
ax.tick_params(axis="y", length=0, labelsize=10, colors="#334155", pad=8)
ax.grid(True, alpha=0.15, linestyle="--", axis="x")
for spine in ax.spines.values():
    spine.set_visible(False)

plt.tight_layout()
plt.show()

# ============================================================
# Comparação de p-valores: completo vs subamostra vs Brown-Forsythe
# (todos os p-valores são 0.0 com 590K linhas - o gráfico prova o ponto)
# ============================================================

fig, ax = plt.subplots(figsize=(12, 7))

x = np.arange(len(var_pd))
width = 0.25

# Adicionar epsilon para log scale conseguir renderizar zeros
epsilon = 1e-300
ax.barh(x - width, var_pd["levene_p_full"] + epsilon, width, label="Levene (dados completos)", color="#3B82F6", alpha=0.8)
ax.barh(x, var_pd["levene_p_subsample"] + epsilon, width, label="Levene (subamostra 5k)", color="#F59E0B", alpha=0.8)
ax.barh(x + width, var_pd["brown_forsythe_p"] + epsilon, width, label="Brown-Forsythe (completo)", color="#E5484D", alpha=0.8)

ax.set_yticks(x)
ax.set_yticklabels(var_pd["variable"])
ax.invert_yaxis()
ax.set_xscale("log")
ax.axvline(x=0.05, color="black", linestyle="--", linewidth=1, alpha=0.5)
ax.text(0.05, len(var_pd) - 0.5, "  p=0.05", fontsize=9, color="#64748B", va="top")

ax.set_title(
    "P-valores do Teste de Levene: Todos Zero com 590K Linhas",
    loc="left",
    fontsize=15,
    color="#0F172A",
    pad=24
)
ax.text(
    0,
    1.02,
    "Todos os p-valores = 0.0 (completo, subamostra, Brown-Forsythe). O tamanho da amostra torna o teste não informativo - use a razão de variâncias.",
    transform=ax.transAxes,
    fontsize=9,
    color="#64748B"
)
ax.set_xlabel("P-valor (escala log)", fontsize=11, color="#64748B")
ax.legend(loc="lower right", fontsize=9, frameon=False)
ax.grid(True, alpha=0.15, linestyle="--", axis="x")
for spine in ax.spines.values():
    spine.set_visible(False)

plt.tight_layout()
plt.show()

### Investigação das Variáveis com IV > 0.5

**Resumo da análise de variância:**

- **18 das 19 variáveis** com IV > 0,5 apresentam razão de variância > 3 (diferença substancial). Apenas V187 tem uma diferença moderada (2,21x)
- Transações fraudulentas apresentam variância **muito maior** que as legítimas. Exemplo extremo: V200 tem 200x mais variância em casos de fraude
- **Todos os p-valores do teste de Levene são 0,0** — dados completos, subamostra de 5k e Brown-Forsythe. Isso confirma a sensibilidade excessiva ao tamanho da amostra: com 590K linhas, o teste não consegue distinguir diferenças reais de diferenças triviais
- A **razão de variância é o critério confiável**, não o p-valor. Todas as 18 variáveis com razão > 3 têm diferenças genuínas e grandes na dispersão dos dados entre fraude e legítimo
- O padrão explica o alto IV: transações fraudulentas têm valores muito mais dispersos/extremos nessas variáveis, enquanto as legítimas estão agrupadas. Essa separação clara é o que gera o alto poder preditivo
- **Correlação entre as 19 variáveis:** 17 pares têm |correlação| > 0,8, indicando alta redundância. Mais correlacionadas: V244-V242 (r=0,975), V199-V190 (r=0,956), V201-V200 (r=0,942)
- **Taxa de nulos:** todas as 19 variáveis têm 0% de valores nulos

**Implicação para o modelo:** apenas uma variável representativa deve ser selecionada de cada grupo de alta correlação para evitar multicolinearidade, priorizando aquelas com maior IV individual.

### Seleção de Variáveis

### Engenharia de Variáveis